In [1]:
pip install multiversity

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 57.9 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 665.8/665.8 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.3/321.3 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 81.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.5 MB/s eta 0:00:00
  Created wheel for owlready2: filename=owlready2-0.49-py3-none-any.whl size=23742423 sha256=15138cee92368051ee79f0d6251107b7dbe72c11df9f0c990eb80b90b55b455f
  Stored in directory: /root/.cache/pip/wheels/43/fe/dc/a0de3c289cfd5923ece6524469d328950e14fa0c90b1088ffa
Successfully built owlready2
Note: you may need to restart the kernel to use updated packages.


In [2]:
from multiversity.multicare_dataset import MedicalDatasetCreator

# 1. Setup the creator (Point this to where you want the files)
mdc = MedicalDatasetCreator(directory='/kaggle/working/')

# 2. Define Filters: We want "Lung Cancer" cases that are "CT Scans"
# Define your detailed filters
lung_cancer_filters = [
    # 1. Broadly search for ANY lung-related case first
    {'field': 'case_strings', 
     'string_list': ['lung', 'pulmonary', 'chest', 'respiratory'], 
     'operator': 'any'},
    
    # 2. STRICTLY filter for CT scans (exclude X-rays to keep data clean)
    {'field': 'label', 'string_list': ['ct', 'computed tomography'], 'operator': 'any'}
]

# Create the Dataset with "Medical Dictionary" Filters
mdc.create_image_classification_dataset(
    dataset_dict={
        'dataset_name': 'MultiCaRe_Lung_v1',
        'common_filter_list': lung_cancer_filters,
        'class_subsets': [
             # --- CLASS 1: MALIGNANT (The "Bad" Cancer) ---
             # We list every specific type of lung cancer here so we don't miss any.
             {'class': 'malignant', 
              'filter_list': [{'field': 'caption', 
                               'string_list': [
                                   'malignant', 'cancer', 'carcinoma', 'tumor', 
                                   'adenocarcinoma',       # Common Type 1
                                   'squamous cell',        # Common Type 2
                                   'small cell',           # Aggressive Type
                                   'metastasis',           # Spread from elsewhere
                                   'metastatic',
                                   'neoplasm',             # Medical term for "new growth" (usually cancer)
                                   'mass',                 # Large tumors are often called "masses"
                                   'nodule'                # Be careful: Nodules can be benign too, but "malignant nodule" is caught by 'malignant'
                               ], 
                               'operator': 'any'}]},

             # --- CLASS 2: BENIGN (The "Safe" Lumps) ---
             # Specific names of non-cancerous lumps.
             {'class': 'benign', 
              'filter_list': [{'field': 'caption', 
                               'string_list': [
                                   'benign', 
                                   'granuloma',            # Inflammatory lump
                                   'hamartoma',            # Fatty lump
                                   'calcified',            # Hardened lump (usually safe)
                                   'abscess',              # Infection pocket (not cancer)
                                   'cyst',                 # Fluid sac
                                   'inflammation'
                               ], 
                               'operator': 'any'}]},

             # --- CLASS 3: NORMAL (Healthy Lungs) ---
             # Captions that say the lung looks fine.
             {'class': 'normal', 
              'filter_list': [{'field': 'caption', 
                               'string_list': [
                                   'normal', 
                                   'unremarkable',         # Doctor speak for "nothing wrong"
                                   'clear',                # "Clear lung fields"
                                   'healthy',
                                   'no abnormality'
                               ], 
                               'operator': 'any'}]}
        ]
    },
    keep_label_columns=False
)

Importing and pre-processing the main files.
Done!
The MultiCaRe_Lung_v1_malignant was successfully created!
The MultiCaRe_Lung_v1_benign was successfully created!
The MultiCaRe_Lung_v1_normal was successfully created!


In [9]:
lung_ct_context_filters = [
    # Search for lung/chest context in the case report
    {'field': 'case_strings', 
     'string_list': ['lung', 'pulmonary', 'chest', 'respiratory'], 
     'operator': 'any'},
    
    # STRICTLY filter for CT scans (exclude X-rays/Ultrasound)
    {'field': 'label', 'string_list': ['ct', 'computed tomography'], 'operator': 'any'}
]
print("--- Step 2: Generating Multimodal Dataset (Full Text) ---")
# This creates 'cases.csv' containing the full patient history.
# Use this for your GPT-2/SLM Branch.
mdc.create_dataset(
    dataset_name='MultiCaRe_Lung_Multimodal',
    filter_list=lung_ct_context_filters,  # Use the same broad filters
    dataset_type='multimodal'
)

# print("Done! Data is ready in './my_thesis_data/'")

--- Step 2: Generating Multimodal Dataset (Full Text) ---
The MultiCaRe_Lung_Multimodal was successfully created!


In [18]:
import pandas as pd

read_cases = pd.read_csv('/kaggle/working/MultiCaRe_Lung_Multimodal/cases.csv')

In [22]:
read_cases.head()

,case_id,pmcid,gender,age,case_text
0,PMC6381877_01,PMC6381877,Male,60.0,A 60-year-old male smoker presented with persi...
1,PMC5912312_01,PMC5912312,Female,41.0,A 41-year-old female with a past medical histo...
2,PMC5912312_03,PMC5912312,Female,34.0,A 34-year-old African American female with a r...
3,PMC5912312_04,PMC5912312,Female,45.0,"A 45-year-old female, professional marathon ru..."
4,PMC5912312_05,PMC5912312,Female,49.0,A 49-year-old female with stage IV sarcoidosis...


In [24]:
img_json = pd.read_json('/kaggle/working/MultiCaRe_Lung_Multimodal/image_metadata.json')
img_json

ValueError: Trailing data

In [ ]:
import matplotlib.pyplot as plt

plt.imread('')

In [27]:
read_cases

,case_id,pmcid,gender,age,case_text
0,PMC6381877_01,PMC6381877,Male,60.0,A 60-year-old male smoker presented with persi...
1,PMC5912312_01,PMC5912312,Female,41.0,A 41-year-old female with a past medical histo...
2,PMC5912312_03,PMC5912312,Female,34.0,A 34-year-old African American female with a r...
3,PMC5912312_04,PMC5912312,Female,45.0,"A 45-year-old female, professional marathon ru..."
4,PMC5912312_05,PMC5912312,Female,49.0,A 49-year-old female with stage IV sarcoidosis...
...,...,...,...,...,...
33758,PMC11747560_01,PMC11747560,Female,54.0,A 54-year-old woman with a medical history of ...
33759,PMC11699715_01,PMC11699715,Female,33.0,We report on a 33-year-old female patient with...
33760,PMC11755523_01,PMC11755523,Male,5.0,"In this study, we present a 5-year-old male ch..."
33761,PMC11663982_01,PMC11663982,Unknown,34.0,"A preterm neonate (gestational age 34 weeks, b..."


In [26]:
import zipfile
import os

# Get all folders in /kaggle/working that start with "MultiCaRe"
working_dir = '/kaggle/working'
multicare_folders = [f for f in os.listdir(working_dir) 
                     if os.path.isdir(os.path.join(working_dir, f)) 
                     and f.startswith('MultiCaRe')]

print(f"Found {len(multicare_folders)} MultiCaRe folders:")
for folder in multicare_folders:
    print(f"  - {folder}")

# Create a zip file for each MultiCaRe folder
for folder in multicare_folders:
    folder_path = os.path.join(working_dir, folder)
    zip_path = os.path.join(working_dir, f"{folder}.zip")
    
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                # Keep the folder structure inside the zip
                arcname = os.path.relpath(file_path, working_dir)
                zipf.write(file_path, arcname=arcname)
    
    print(f"✓ Created {folder}.zip")

print("\nAll MultiCaRe folders have been zipped!")


Found 5 MultiCaRe folders:
  - MultiCaRe_Lung_v1_benign
  - MultiCaRe_Lung_v1_malignant
  - MultiCaRe_Lung_Multimodal
  - MultiCaRe_Lung_v1
  - MultiCaRe_Lung_v1_normal
✓ Created MultiCaRe_Lung_v1_benign.zip
✓ Created MultiCaRe_Lung_v1_malignant.zip
✓ Created MultiCaRe_Lung_Multimodal.zip
✓ Created MultiCaRe_Lung_v1.zip
✓ Created MultiCaRe_Lung_v1_normal.zip

All MultiCaRe folders have been zipped!
